# Position gives order. Keywords give names.

**Python Foundation · An interactive experiment**

Choose **Runtime → Run all** in Google Colab, then choose a call or edit it in the explorer. All code cells are editable and run real Python. Expected errors are caught so that Run all can finish.

Based on the eight function calls in `untitled.ipynb`.

```python
def receive_variables(X, Y, Z):
    return (X, Y, Z)
```

`X`, `Y`, and `Z` are **parameters** in the definition. The values supplied by a call are **arguments**. Here, each parameter accepts either a positional argument or a keyword argument.


In [1]:
def receive_variables(X, Y, Z):
    return (X, Y, Z)

print(receive_variables(X=1, Y=2, Z=3))
print(receive_variables(1, 2, 3))
print(receive_variables(1, 2, Z=3))


(1, 2, 3)
(1, 2, 3)
(1, 2, 3)


## Match the values to the parameters

For this function:

1. Positional arguments fill `X`, then `Y`, then `Z`, from left to right.
2. Keyword arguments choose a parameter by its exact name. Their order can change.
3. Each required parameter must receive exactly one value.
4. In these ordinary calls, positional arguments must appear before keyword arguments.

The returned tuple always follows the function body: `(X, Y, Z)`. Reordering keywords does not reorder that tuple.

| Call | Binding | Result |
|---|---|---|
| `receive_variables(X=1, Y=2, Z=3)` | X=1, Y=2, Z=3 | `(1, 2, 3)` |
| `receive_variables(1, 2, 3)` | X=1, Y=2, Z=3 | `(1, 2, 3)` |
| `receive_variables(1, 2, Z=3)` | X=1, Y=2, Z=3 | `(1, 2, 3)` |
| `receive_variables(2, 3, X=1)` | Positions already fill X=2, Y=3; X=1 conflicts | `TypeError` |
| `receive_variables(1, Y=2, 3)` | A positional argument follows a keyword | `SyntaxError` |
| `receive_variables(3, X=1, Y=2)` | Position already fills X=3; X=1 conflicts | `TypeError` |
| `receive_variables(Y=2, X=1, Z=3)` | X=1, Y=2, Z=3 | `(1, 2, 3)` |
| `receive_variables(1, Z=3, Y=2)` | X=1, Y=2, Z=3 | `(1, 2, 3)` |

Python does not skip `X` when it sees a later `X=1`. In `receive_variables(2, 3, X=1)`, the first positional argument already belongs to `X`.


In [2]:
original_calls = [
    "receive_variables(X=1, Y=2, Z=3)",
    "receive_variables(1, 2, 3)",
    "receive_variables(1, 2, Z=3)",
    "receive_variables(2, 3, X=1)",
    "receive_variables(1, Y=2, 3)",
    "receive_variables(3, X=1, Y=2)",
    "receive_variables(Y=2, X=1, Z=3)",
    "receive_variables(1, Z=3, Y=2)",
]

# Compile each string separately: invalid syntax cannot be caught by placing
# the invalid expression directly inside a try block in the same code cell.
for source in original_calls:
    print(source)
    try:
        code = compile(source, "<argument-example>", "eval")
        result = eval(code, {"__builtins__": {}, "receive_variables": receive_variables})
        print("  Result:", result)
    except (SyntaxError, TypeError) as error:
        print(f"  {type(error).__name__}: {getattr(error, 'msg', str(error))}")


receive_variables(X=1, Y=2, Z=3)
  Result: (1, 2, 3)
receive_variables(1, 2, 3)
  Result: (1, 2, 3)
receive_variables(1, 2, Z=3)
  Result: (1, 2, 3)
receive_variables(2, 3, X=1)
  TypeError: receive_variables() got multiple values for argument 'X'
receive_variables(1, Y=2, 3)
  SyntaxError: positional argument follows keyword argument
receive_variables(3, X=1, Y=2)
  TypeError: receive_variables() got multiple values for argument 'X'
receive_variables(Y=2, X=1, Z=3)
  Result: (1, 2, 3)
receive_variables(1, Z=3, Y=2)
  Result: (1, 2, 3)


## Two failures, two stages

**SyntaxError:** `receive_variables(1, Y=2, 3)` cannot be compiled. Move the final positional argument before the keywords, or name it: `receive_variables(1, Y=2, Z=3)`.

**TypeError:** `receive_variables(2, 3, X=1)` has valid syntax, but `X` receives two values. To request X=1, Y=2, Z=3, use `receive_variables(1, 2, 3)` or `receive_variables(Y=2, Z=3, X=1)`.

Neither failing call enters the function body. A keyword never overwrites an earlier positional argument.


In [3]:
print(receive_variables(1, Y=2, Z=3))
print(receive_variables(Y=2, Z=3, X=1))

# Position changes the binding; keyword order does not.
print(receive_variables(3, 2, 1))
print(receive_variables(Z=3, Y=2, X=1))


(1, 2, 3)
(1, 2, 3)
(3, 2, 1)
(1, 2, 3)


## Explore the binding

Choose any original call, or type a custom call such as `receive_variables(10, Z=30, Y=20)`. The diagram shows which argument supplies each parameter and highlights duplicate assignments.

Try leaving out `Z`, adding `W=4`, or supplying four positional arguments. The result comes from real Python. The explorer accepts literal values and ordinary arguments; unpacking with `*` or `**` is outside this experiment.


In [4]:
import ast
import html
import inspect
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

def analyze_call(source):
    try:
        tree = ast.parse(source, mode="eval")
        compile(tree, "<argument-explorer>", "eval")
    except SyntaxError as error:
        return {"error": "SyntaxError", "detail": error.msg, "mapping": None}
    call = tree.body
    if not (isinstance(call, ast.Call) and isinstance(call.func, ast.Name)
            and call.func.id == "receive_variables"):
        return {"error": "Explorer input", "detail": "Enter one receive_variables(...) call.", "mapping": None}
    try:
        if any(isinstance(a, ast.Starred) for a in call.args) or any(k.arg is None for k in call.keywords):
            raise ValueError("Use ordinary positional and keyword arguments without unpacking.")
        args = [ast.literal_eval(a) for a in call.args]
        kwargs = {k.arg: ast.literal_eval(k.value) for k in call.keywords}
    except (ValueError, TypeError, SyntaxError):
        return {"error": "Explorer input", "detail": "Use literal values, such as 1, 'hello', or [1, 2], without * or ** unpacking.", "mapping": None}
    names = list(inspect.signature(receive_variables).parameters)
    mapping = {name: [] for name in names}
    for index, value in enumerate(args[:len(names)]):
        mapping[names[index]].append((f"position {index + 1}", value))
    for name, value in kwargs.items():
        if name in mapping:
            mapping[name].append((f"keyword {name}", value))
    try:
        result = receive_variables(*args, **kwargs)
        return {"result": result, "mapping": mapping}
    except TypeError as error:
        return {"error": "TypeError", "detail": str(error), "mapping": mapping}

def render_experiment(source):
    info = analyze_call(source)
    escape = lambda value: html.escape(str(value))
    cards = []
    if info["mapping"] is not None:
        for name, sources in info["mapping"].items():
            conflict = len(sources) > 1
            color = "#b42318" if conflict else "#167a96" if sources else "#64748b"
            values = "<br>".join(f"{escape(origin)} → <b>{escape(repr(value))}</b>" for origin, value in sources) or "No value supplied"
            note = "<br><strong>Two values: conflict</strong>" if conflict else ""
            cards.append(f'<div style="flex:1;min-width:150px;padding:16px;border:1px solid {color};border-radius:10px"><strong style="font-size:22px;color:{color}">{name}</strong><p>{values}{note}</p></div>')
    if "error" in info:
        result = f'<strong style="color:#b42318">{escape(info["error"])}</strong>: {escape(info["detail"])}'
        detail = "The function body is not entered." if info["error"] in ("SyntaxError", "TypeError") else ""
    else:
        result = f'<strong style="color:#167a96">Returned tuple: {escape(repr(info["result"]))}</strong>'
        detail = "The return statement reads X, then Y, then Z."
    return f'<div style="padding:22px;border:1px solid #cbd5e1;border-radius:14px;background:#fff;color:#172b3a;font:15px/1.7 system-ui"><code style="overflow-wrap:anywhere">{escape(source)}</code><p>Parameters: <b>receive_variables(X, Y, Z)</b></p><div style="display:flex;flex-wrap:wrap;gap:12px">{"".join(cards)}</div><p style="overflow-wrap:anywhere">{result}</p><p>{detail}</p></div>'

examples = widgets.Dropdown(options=original_calls, description="Example:", layout=widgets.Layout(width="95%"))
call_input = widgets.Text(value=original_calls[0], description="Call:", continuous_update=False, layout=widgets.Layout(width="95%"))
reset_button = widgets.Button(description="Reset")
view = widgets.HTML()

def update(change=None):
    view.value = render_experiment(call_input.value)

def select_example(change):
    call_input.value = change["new"]

def reset(button):
    examples.value = original_calls[0]
    call_input.value = original_calls[0]
    update()

examples.observe(select_example, names="value")
call_input.observe(update, names="value")
reset_button.on_click(reset)
update()
display(widgets.VBox([examples, call_input, reset_button, view]))


## Predict, then run

1. What tuple does `receive_variables(3, Z=1, Y=2)` return?
2. Does `receive_variables(Z=1, X=3, Y=2)` return the same tuple?
3. Does `receive_variables(3, X=1, Z=2)` overwrite X, or raise an error?

Edit the calls below and check your predictions.


In [5]:
print(receive_variables(3, Z=1, Y=2))
print(receive_variables(Z=1, X=3, Y=2))
try:
    print(receive_variables(3, X=1, Z=2))
except TypeError as error:
    print(f"TypeError: {error}")


(3, 2, 1)
(3, 2, 1)
TypeError: receive_variables() got multiple values for argument 'X'


## Remember

| Argument style | How a value finds its parameter |
|---|---|
| Positional: `1, 2, 3` | By position in the function definition |
| Keyword: `Z=3, X=1, Y=2` | By parameter name |
| Mixed: `1, Z=3, Y=2` | Positions first, then names for the remaining parameters |

**Position gives order. Keywords give names. Each parameter gets one value.**

These rules describe the ordinary calls above. Python also supports positional-only parameters (`/`), keyword-only parameters (`*`), defaults, and argument unpacking, which are separate topics.

Further reading: [Python keyword arguments](https://docs.python.org/3/tutorial/controlflow.html#keyword-arguments).
